In [0]:
/Volumes/workspace/injestion_layer/injestion_layer_volume/inventory_data.json
/Volumes/workspace/injestion_layer/injestion_layer_volume/orders_data.csv
/Volumes/workspace/injestion_layer/injestion_layer_volume/returns_data.xlsx

In [0]:
# Load messy raw data into DataFrames from CSVs
# df_orders_raw = spark.read.parquet("/Volumes/workspace/injestion_layer/injestion_layer_volume/orders_data.csv")
# df_returns_raw = spark.read.parquet("/Volumes/workspace/injestion_layer/injestion_layer_volume/returns_data.xlsx")
# df_inventory_raw = spark.read.parquet("/Volumes/workspace/injestion_layer/injestion_layer_volume/inventory_data.json")


In [0]:
import pandas as pd

# Load JSON using Spark
pandas_inventory = pd.read_json("/Volumes/workspace/injestion_layer/injestion_layer_volume/inventory_data.json")
injestion_inventory = spark.createDataFrame(pandas_inventory)

# Load Excel using pandas, then convert to Spark DataFrame
pandas_returns = pd.read_excel("/Volumes/workspace/injestion_layer/injestion_layer_volume/returns_data.xlsx")
injestion_returns = spark.createDataFrame(pandas_returns)

# Load CSV using Spark with header option
injestion_order = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/injestion_layer/injestion_layer_volume/orders_data.csv")


In [0]:
# display(injestion_inventory)
# display(injestion_order)
# display(injestion_returns)

In [0]:
# Create managed tables (Delta format required for Unity Catalog)
# injestion_inventory.write.format("delta").mode("overwrite").saveAsTable("bronze_inventory_ingestion")

injestion_returns.write.format("parquet").mode("overwrite").saveAsTable("bronze_return_ingestion")

# injestion_order.write.format("delta").mode("overwrite").saveAsTable("bronze_order_ingestion")

In [0]:
# Extract first row as header
first_row = injestion_order.first()
columns = [str(item).strip() for item in first_row]

# Remove the first row (header row now part of data)
injestion_order = injestion_order.rdd.zipWithIndex() \
    .filter(lambda x: x[1] > 0) \
    .map(lambda x: x[0]) \
    .toDF(columns)

# Show cleaned data
display(injestion_order)

In [0]:
# Save raw data to Bronze tables without any transformations
injestion_order.write.mode("overwrite").format("delta").saveAsTable("bronze_orders")
# injestion_return.write.mode("overwrite").format("delta").saveAsTable("bronze_returns")
# injestion_inventory.write.mode("overwrite").format("delta").saveAsTable("bronze_inventory")

In [0]:


# 🥈 SILVER LAYER – Clean & Normalize
# 🔹 Cleaning Orders Data

from pyspark.sql.functions import *

df_orders = (
    df_orders_raw
    # Fix inconsistent column names
    .withColumnRenamed("Order iD", "Order_ID")
    .withColumnRenamed("order_date", "Order_Date")

    # Convert Order_Date to proper format, replacing "/" with "-" and casting to date
    .withColumn("Order_Date", to_date(regexp_replace(col("Order_Date"), "/", "-"), "dd-MM-yyyy"))

    # Standardize Customer ID format – uppercase and trimmed
    .withColumn("CustomerID", trim(upper(col("Customer_ID"))))

    # Normalize customer names to title case
    .withColumn("Customer_Name", initcap(trim(col("Customer_Name"))))

    # Clean emails – lowercase and trimmed
    .withColumn("Email", lower(trim(col("Email"))))

    # Handle empty/null payment modes
    .withColumn("Payment_Mode", when(length(trim(col("Payment_Mode"))) == 0, "unknown").otherwise(col("Payment_Mode")))

    # Replace null promo codes with placeholder
    .withColumn("Promo_Code", coalesce(col("Promo_Code"), lit("NO_PROMO")))

    # Fix negative order amounts and cast to double
    .withColumn("Order_Amount", abs(col("Order_Amount").cast("double")))

    # Add year and month columns for easier reporting
    .withColumn("Order_Year", year("Order_Date"))
    .withColumn("Order_Month", month("Order_Date"))

    # Create unique hash key for deduplication or tracking
    .withColumn("Order_Hash", sha2(concat_ws("|", *df_orders_raw.columns), 256))

    # Drop rows where critical fields are missing
    .dropna(subset=["Order_ID", "Order_Amount", "CustomerID"])

    # Remove duplicate Order IDs
    .dropDuplicates(["Order_ID"])
)

# Save cleaned Orders data
df_orders.write.mode("overwrite").format("delta").saveAsTable("silver_orders")



# 🔹 Cleaning Inventory Data

df_inventory = (
    df_inventory_raw
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")
    
    # 2. Clean stock column: convert to integer
    .withColumn("Stock", 
        when(col("stock").rlike("^[0-9]+$"), col("stock").cast(IntegerType()))  # Numeric values
        .when(col("stock").isNull() | (col("stock") == ""), lit(None))  # Null or blank
        .otherwise(
            when(col("stock").rlike(".*twenty five.*"), lit(25))
            .when(col("stock").rlike(".*twenty.*"), lit(20))
            .when(col("stock").rlike(".*eighteen.*"), lit(18))
            .when(col("stock").rlike(".*fifteen.*"), lit(15))
            .when(col("stock").rlike(".*twelve.*"), lit(12))
            .otherwise(lit(None))
        ).cast(IntegerType())
    )
    
    # 3. Clean LastStocked: normalize multiple date formats to yyyy-MM-dd
    .withColumn("LastStocked", to_date(
        regexp_replace("LastStocked", "[./]", "-"), "yyyy-MM-dd"
    ))
    
    # 4. Clean CostPrice: extract numeric value and convert to float
    .withColumn("CostPrice", 
        regexp_extract(col("CostPrice"), r"(\d+\.?\d*)", 1).cast(DoubleType())
    )

    # 5. Clean Warehouse: remove special characters, trim, capitalize first letter
    .withColumn("Warehouse", 
        initcap(trim(regexp_replace(col("warehouse"), r"[^a-zA-Z0-9\s]", " ")))
    )
    
    # 6. Standardize Available: convert to boolean
    .withColumn("Available", 
        when(lower(col("available")).isin("yes", "y", "true"), lit(True))
        .when(lower(col("available")).isin("no", "n", "false"), lit(False))
        .otherwise(None)
    )
    
    # 7. Drop raw messy columns
    .drop("stock", "warehouse", "available")
)

# Display the cleaned data
df_inventory_cleaned.display()

# Optional: Save to Silver Layer
df_inventory_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_inventory")


# 🔹 Cleaning Returns Data

df_returns = (

   df_return_raw
    # 2.1 Standardize column names (if needed)
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")
    
    # 2.2 Clean ReturnDate → standardize date formats
    .withColumn("ReturnDate", to_date(
        regexp_replace("ReturnDate", r"[./]", "-"), "dd-MM-yyyy"
    ))
    
    # 2.3 Clean RefundStatus → lowercase, remove special characters
    .withColumn("RefundStatus", lower(regexp_replace(col("RefundStatus"), r"[^a-zA-Z]", "")))
    
    # 2.4 Clean ReturnAmount → extract numeric part regardless of currency
    .withColumn("ReturnAmount", 
        regexp_extract(col("ReturnAmount"), r"(\d+\.?\d*)", 1).cast(DoubleType())
    )
    
    # 2.5 Clean PickupAddress → remove special characters
    .withColumn("PickupAddress", initcap(trim(regexp_replace(col("PickupAddress"), r"[^a-zA-Z0-9\s]", " "))))
    
    # 2.6 Clean Product → remove extra symbols and spaces
    .withColumn("Product", initcap(trim(regexp_replace(col("Product"), r"[^a-zA-Z0-9\s]", ""))))
    
    # 2.7 Clean CustomerID → trim, fix wrong prefixes
    .withColumn("CustomerID", trim(upper(col("CustomerID"))))
    
    # 2.8 Drop rows with null ReturnID (R014)
    .filter(col("ReturnID").isNotNull())
)

# Step 3: Show cleaned Silver data
df_return_cleaned.display()

# Step 4: Save to Silver Delta Table
df_return_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_returns")





In [0]:
🥇 GOLD LAYER – Aggregation & KPIs
🔸 Enrich Orders with Returns & Inventory

from pyspark.sql.functions import *

# Load Silver Tables
df_orders = spark.read.table("silver_orders")
df_returns = spark.read.table("silver_returns")
df_inventory = spark.read.table("silver_inventory")

# 🟡 STEP 1: Join Orders with Returns (LEFT JOIN to retain all orders)
df_order_return = df_orders.join(df_returns, on="OrderID", how="left")

# 🟡 STEP 2: Join Inventory (LEFT JOIN on cleaned Product_Name)
df_enriched = df_order_return.join(
    df_inventory,
    df_order_return.ProductName == df_inventory.ProductName,
    "left"
)

# 🟡 STEP 3: KPI Aggregations at Product Level
df_kpi = (
    df_enriched.groupBy("ProductName")
    .agg(
        count("OrderID").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers"),
        count("ReturnID").alias("Total_Returns"),
        round((count("ReturnID") / count("OrderID")) * 100, 2).alias("Return_Rate(%)"),
        round(sum("OrderAmount"), 2).alias("Total_Revenue"),
        round(avg("OrderAmount"), 2).alias("Avg_Order_Value"),
        sum("Stock").alias("Total_Stock"),
        round(avg("CostPrice"), 2).alias("Avg_Cost"),
        round(sum("OrderAmount") - (sum("Stock") * avg("CostPrice")), 2).alias("Net_Profit")
    )
)

# 🟡 STEP 4: Save to Gold Table
df_kpi.write.mode("overwrite").format("delta").saveAsTable("gold_product_kpis")

